In [1]:
import os
import sys
import multiprocessing
from tqdm import tqdm


In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from llm_models.code_llms import Mistral
from mcq_inconsistency.mcq_inconsistency_tester import LLMMCQInconsistencyTester
from mcq_inconsistency.prompt_templates.prompt_template import MCQInconsistencyPromptTemplate
from utility.constants import CodeMMLU, LexicalMutations, SyntacticMutations, LogicalMutations
from mcq_inconsistency.utility.codemmlu_helper import CodeGenerationCodeMMLUHelper

In [4]:
FOR2WHILE = SyntacticMutations.FOR2WHILE
FOR2ENUMERATE = SyntacticMutations.FOR2ENUMERATE

RANDOM_MUTATION = LexicalMutations.RANDOM
SEQUENTIAL_MUTATION = LexicalMutations.SEQUENTIAL
LITERAL_FORMAT = LexicalMutations.LITERAL_FORMAT

BOOLEAN_LITERAL = LogicalMutations.BOOLEAN_LITERAL
DEMORGAN = LogicalMutations.DEMORGAN
COMMUTATIVE_REORDER = LogicalMutations.COMMUTATIVE_REORDER
CONSTANT_UNFOLD = LogicalMutations.CONSTANT_UNFOLD
CONSTANT_UNFOLD_ADD = LogicalMutations.CONSTANT_UNFOLD_ADD
CONSTANT_UNFOLD_MULT = LogicalMutations.CONSTANT_UNFOLD_MULT

In [5]:
task_set = "CodeMMLU_MCQ_code_completion"
llmtester = LLMMCQInconsistencyTester(task_set)

MongoDB connected


In [6]:
def check_prog_validity(full_sol, check, func_name):
    try:
        multiprocessing_queue = multiprocessing.Queue()

        verify_answer_process = multiprocessing.Process(        
        target= CodeGenerationCodeMMLUHelper.run_llm_answer,
        args = (full_sol, check, func_name, multiprocessing_queue)
        )
        
        verify_answer_process.start()
        verify_answer_process.join(timeout=5)

        if verify_answer_process.is_alive():
            verify_answer_process.kill()
            verify_answer_process.join()
            raise RuntimeError("The mutated answer took too long to run, which could inidicate some sort of infinite loop")

        if not multiprocessing_queue.empty():
            error = multiprocessing_queue.get()
            raise error

    except Exception as e:
        raise e

In [12]:
num_tests = llmtester.question_database.count_documents({})

ANS_DICT = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3
}

options_dict = {}

passed_dict = {}

for i in tqdm(
    range(num_tests)
    ):
    task_id = f"CodeMMLU{i}"
    sample_qn = llmtester.question_database.find_one({"_id":task_id})

    choices : dict = sample_qn['choices']
    check = sample_qn['check']
    answer = sample_qn['answer']
    func_name = sample_qn['func_name']
    question = sample_qn['question']

    correct_choice = choices[answer]

    full_canonical_sol = question + "\n" + correct_choice

    check_prog_validity(
        full_sol=full_canonical_sol,
        check= check,
        func_name=func_name
    )
    invalid = 0
    for key, choice in choices.items():
        if key == answer:
            continue

        test_sol = question + "\n" + choice

        try:
            check_prog_validity(
                full_sol=test_sol,
                check= check,
                func_name=func_name
            )
        except RuntimeError as e:
            print(f"{task_id}: ran for too long")
            print(f"{choice}")
            continue

        except Exception:
            continue

        passed_test = passed_dict.get(task_id, [])
        passed_dict[task_id] = passed_test + [key]

    num_options = len(choices) - 1 -invalid 
    
    if num_options <= 1:
        print(f"{task_id} has {num_options} options only" )

    options_dict[num_options] = options_dict.get(num_options, 0) + 1
    

  3%|▎         | 5/164 [00:01<00:31,  5.04it/s]

CodeMMLU4 has 1 options only


 21%|██▏       | 35/164 [00:07<00:24,  5.31it/s]

CodeMMLU35 has 0 options only


 34%|███▎      | 55/164 [00:11<00:24,  4.44it/s]

CodeMMLU55: ran for too long
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fib(n - 3) + fib(n - 2) 


 95%|█████████▌| 156/164 [00:31<00:01,  5.20it/s]

CodeMMLU156: ran for too long
    num = [1, 4, 5, 9, 10, 40, 50, 90, 100, 400, 500, 900, 1000]
    sym = ["I", "IV", "V", "IX", "X", "XL", "L", "XC", "C", "CD", "D", "CM", "M"]
    i = 12
    res = ''
    while number:
        div = number // num[i]
        number %= num[i]
        res += sym[i] * div  
    return res.lower()


100%|██████████| 164/164 [00:38<00:00,  4.23it/s]


In [10]:
for key, val in passed_dict.items():
    print(key, ":", val)

In [ ]:
for key, val in options_dict.items():
    print(key, ":", val)

2 : 8
3 : 154
1 : 1
0 : 1
